# GRT · Colab 사후 RTLA 분석

GPU 런타임(T4 권장)을 선택하고 위에서 아래로 실행합니다. **학습 없이 현재 WikiText GRT checkpoint 하나를 FP32·단일 GPU에서 분석**합니다. Kaggle T4×2에서 학습한 checkpoint도 사용할 수 있습니다. W&B key는 필요하지 않습니다.

Google Drive에 다음처럼 넣어두세요. Kaggle의 해당 run에서 `grt/best.pt`를 복사하고, 학습에 사용한 prepared 폴더에서 아래 파일만 복사하면 됩니다. `last.pt`나 보존한 중간 checkpoint도 선택할 수 있습니다.

```text
MyDrive/GRT_analysis/
├── checkpoints/
│   ├── m10/best.pt
│   ├── m40/best.pt
│   └── m160/best.pt
└── wikitext103_gpt2_bpe/
    ├── manifest.json
    ├── validation.pt
    └── tokenizer/             # manifest에 기록된 원본 파일 전부
```

`train.pt`, `test.pt`, optimizer용 별도 파일, `experiment_report.json`은 입력에 필요하지 않습니다. 전체 `best.pt`에는 optimizer가 들어 있어도 분석 시 사용하지 않습니다. **prepared 데이터를 새로 만들지 않고 원본을 복사**합니다. checkpoint의 manifest와 validation/tokenizer hash가 다르면 실행을 중단합니다.

[사용 안내·사후 결과표](https://github.com/YeoJune/GRT/blob/main/docs/20261014_posthoc_analysis.md)

GitHub 반영 전 코드 묶음을 사용할 때는 이 노트북과 함께 받은 **`grt-colab-analysis.zip`을 Colab 왼쪽 파일 탭의 `/content`에 업로드한 뒤** 아래 셀을 실행하세요. ZIP의 분석 코드를 설치합니다. ZIP은 실행 코드이며 checkpoint/data와 별도입니다.


## 1. 코드·라이브러리 준비

최신 main과 분석 helper를 설치합니다. 같은 런타임에서 코드를 갱신한 뒤 이미 `grt`를 import했다면 런타임을 재시작하고 다시 실행하세요.


In [ ]:
from pathlib import Path
import hashlib, shutil, subprocess, sys

bundle = Path('/content/grt-colab-analysis.zip')
if bundle.is_file():
    tag = hashlib.sha256(bundle.read_bytes()).hexdigest()[:10]
    repo = Path('/content') / ('GRT_analysis_source_' + tag)
    if not repo.exists():
        shutil.unpack_archive(str(bundle), str(repo))
    print('업로드한 분석 코드 사용:', repo)
else:
    repo = Path('/content/GRT')
    if not (repo / '.git').exists():
        subprocess.run(['git', 'clone', '--branch', 'main', '--single-branch',
                        'https://github.com/YeoJune/GRT.git', str(repo)], check=True)
    else:
        dirty = subprocess.check_output(['git', '-C', str(repo), 'status', '--porcelain'], text=True)
        if dirty.strip():
            print('로컬 수정이 있어 코드 갱신을 생략합니다:', repo)
        else:
            subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only', 'origin', 'main'], check=True)
if not (repo / 'src/grt/utils/posthoc.py').is_file():
    raise RuntimeError('분석 코드가 없습니다. grt-colab-analysis.zip을 /content에 올린 뒤 다시 실행하세요.')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(repo)], check=True)


## 2. Drive 연결

큰 checkpoint는 Drive에 업로드해 두면 세션이 바뀌어도 재사용할 수 있습니다. 왼쪽 파일 탭으로 `/content`에 직접 올리는 경우 `use_drive=False`로 바꾸고 다음 셀의 경로를 수정하세요.


In [ ]:
use_drive = True
if use_drive:
    from google.colab import drive
    drive.mount('/content/drive')


## 3. checkpoint·분석 옵션 선택

**M/MLP/backbone은 checkpoint에서 자동으로 읽습니다.** 비교할 때 `batch_size`, `segments`, `warmup_segments`, `horizon`을 같게 유지하세요. batch를 바꾸면 validation stream 분할과 probe token이 달라집니다.

첫 실행은 `analyze_slots=False`로 reset/swap·retention·gate 분포를 확인합니다. 슬롯별 초기화까지 원하면 `True`로 바꾸고 이 셀부터 다시 실행하세요. M160에서는 슬롯 개입이 `M × 측정 segment 수`만큼 forward를 추가하므로 시간이 더 걸립니다. 옵션 변경은 별도 출력 폴더로 저장됩니다.

기본 warmup3·측정8·batch8은 학습 중 기본 RTLA probe와 맞춥니다. `horizon=4`는 개입 segment부터 이후3 segments까지입니다. 다른 window를 확인하려면 `warmup_segments=11` 등으로 바꾸되 모든 M에 같은 값을 사용하세요. 여기서 gate의 시간축은 **checkpoint를 고정한 상태의 probe segment**이며 학습 step 변화가 아닙니다.


In [ ]:
from grt.utils.posthoc import WikiTextPosthoc

checkpoint = Path('/content/drive/MyDrive/GRT_analysis/checkpoints/m40/best.pt')
prepared_dir = Path('/content/drive/MyDrive/GRT_analysis/wikitext103_gpt2_bpe')
output_dir = Path('/content/drive/MyDrive/GRT_analysis/outputs')

batch_size = 8
segments = 8
warmup_segments = 3
horizon = 4
analyze_slots = False  # 슬롯별 초기화까지 하려면 True

job = WikiTextPosthoc(
    checkpoint=checkpoint, prepared_dir=prepared_dir, output_dir=output_dir,
    device='cuda', batch_size=batch_size, segments=segments,
    warmup_segments=warmup_segments, horizon=horizon, slots=analyze_slots,
)
job.inspect()


## 4. 분석 실행

실제 checkpoint step·M·MLP·데이터 검증 결과와 예상 forward 횟수를 위 출력에서 확인하고 실행합니다. FP32, dropout off, gradient off, TF32 off입니다. 학습 checkpoint·학습 설정 파일을 수정하지 않습니다. 같은 입력·옵션·분석 코드의 완료 결과가 있으면 재사용합니다.

진행 횟수와 대략 남은 시간을 출력합니다. CUDA OOM이면 다른 GPU 프로그램을 종료하세요. batch를 줄여야 한다면 모든 비교 후보를 같은 새 batch로 다시 분석하세요. 세션이 중단돼도 Drive의 checkpoint는 유지되지만 미완료 분석은 처음부터 다시 실행합니다.


In [ ]:
result = job.run()


## 5. 그림 확인

`memory.png`: token 위치·segment별 reset/swap, 슬롯별 초기화, retention.
`trace.png`: slot별 batch 평균 write/state/update와 write pooling attention. read gate off 표시를 읽기 사용량으로 해석하지 않습니다. pooling attention은 token→register 읽기 attention이 아닙니다.

슬롯 분석을 끈 경우 해당 패널은 미측정으로 표시합니다. 실제 example별 gate 분포·포화 비율은 다음 JSON에 포함됩니다.


In [ ]:
from IPython.display import Image, display

print(result['gate_statistics']['quantiles'])
print('near-closed:', result['gate_statistics']['fraction_near_closed'],
      'near-open:', result['gate_statistics']['fraction_near_open'])
for name in ('memory.png', 'trace.png'):
    display(Image(filename=str(job.analysis_dir / name)))


## 6. 보내줄 파일 하나 다운로드

**`posthoc_report.json` 하나만 보내주세요.** checkpoint 식별 hash·step, 원래 학습 설정, 데이터 manifest, 실제 분석 조건·GPU/시간, 정상/reset/swap, early/late, slot/retention, example별 gate 분포와 분석 배열이 함께 들어갑니다. checkpoint·optimizer·원본 token stream은 들어가지 않습니다.

에이전트는 원본을 `results/posthoc/<report_id>.json`에 보관하고 사후 결과표를 채웁니다. M10/M40/M160의 같은 조건·같은 step 분석을 맞춰 slot 기여·gate 포화·retention을 비교하며, 기존 학습 결과와는 checkpoint/데이터 식별값을 확인해 연결합니다. 이전 window나 다른 step은 별도 행으로 기록합니다.

다운로드가 막히면 출력된 Drive 경로에서 JSON을 직접 받으세요. PNG·NPZ는 Drive에 보존되지만 기본 전달에는 필요 없습니다. 실행 중 측정하는 PPL은 고정 probe PPL이며 전체 validation PPL을 대체하지 않습니다.


In [ ]:
job.report(download=True)
